# KishoLens ETL Pipeline Prototype

This notebook implements a streamed ETL pipeline to ingest datasets from Hugging Face and Project Gutenberg, clean the raw text (removing HTML tags, removing translator notes, and handling Japanese ruby tags), save it to SQLite (`data/kisholens.db`), and preview NLP feature extraction.

In [1]:
import os
import re
import unicodedata
import asyncio
import aiohttp
from typing import Optional
from bs4 import BeautifulSoup
from datasets import load_dataset
from sqlmodel import SQLModel, Field, Session, create_engine, select, func

In [2]:
# Declare SQLModel tables

# Clear metadata tables registry to avoid InvalidRequestError when re-running this cell in Jupyter
SQLModel.metadata.clear()

class Novel(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    author: str
    source: str

class Chapter(SQLModel, table=True):
    __table_args__ = {"extend_existing": True}
    id: Optional[int] = Field(default=None, primary_key=True)
    novel_id: int = Field(foreign_key="novel.id")
    chapter_number: int
    title: str
    text_ja: str
    text_en: str
    text_zh: str = Field(default="")

In [3]:
# Text cleaning and parsing methods

def clean_html(text: str) -> str:
    """Removes HTML tags using BeautifulSoup with lxml parser."""
    if not text:
        return ""
    soup = BeautifulSoup(text, "lxml")
    return soup.get_text()

def clean_japanese(text: str) -> str:
    """Removes Japanese ruby tags (｜ and 《 》) and normalizes unicode (NFKC)."""
    if not text:
        return ""
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'《.*?》', '', text)
    text = text.replace('｜', '')
    return text

def clean_english(text: str) -> str:
    """Strips translator/editor notes matching [TL note: ...] or (T/N: ...)."""
    if not text:
        return ""
    pattern = r"(?s)(?:\[(?:TL\s*note|T/N|Editor's\s*note|EN|TN):.*?\]|\((?:TL\s*note|T/N|Editor's\s*note|EN|TN):.*?\))"
    text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    return text.strip()

def parse_chapter_number(title: str) -> Optional[int]:
    """Helper to extract a chapter number from a chapter title string."""
    match_en = re.search(r'(?:[Cc]hapter|[Cc]h)\s*(\d+)', title, re.IGNORECASE)
    if match_en:
        return int(match_en.group(1))
    match_num = re.match(r'^(\d+)', title)
    if match_num:
        return int(match_num.group(1))
    return None

def extract_chapter_info(src: str, trg: str, index: int):
    """
    Extracts chapter number and titles from the first line of raw text.
    If the first line represents a title, it strips it from the returned body text.
    """
    src_stripped = src.strip()
    trg_stripped = trg.strip()
    
    src_split = src_stripped.split('\n', 1)
    src_first_line = src_split[0]
    
    trg_split = trg_stripped.split('\n', 1)
    trg_first_line = trg_split[0]
    
    chapter_number = None
    match_ja = re.match(r'^(\d+)[.．\s]', src_first_line)
    if match_ja:
        chapter_number = int(match_ja.group(1))
    else:
        chapter_number = parse_chapter_number(trg_first_line)
    
    is_header = chapter_number is not None
        
    title_ja = src_first_line
    if match_ja:
        title_ja = src_first_line[match_ja.end():].strip()
    
    title_en = trg_first_line
    match_en_title = re.match(r'^(?:[Cc]?hapter|[Cc]h)\s*\d+[\s:.\-]*', trg_first_line, re.IGNORECASE)
    if match_en_title:
        title_en = trg_first_line[match_en_title.end():].strip()
    else:
        match_en_start_num = re.match(r'^\d+[\s:.\-]*', trg_first_line)
        if match_en_start_num:
            title_en = trg_first_line[match_en_start_num.end():].strip()
            
    if not is_header:
        title_ja = f"Chapter {chapter_number if chapter_number is not None else ''}"
        title_en = f"Chapter {chapter_number if chapter_number is not None else ''}"
        
    body_ja = src
    body_en = trg
    if is_header:
        body_ja = src_split[1] if len(src_split) > 1 else ""
        body_en = trg_split[1] if len(trg_split) > 1 else ""
        
    return chapter_number, title_ja, title_en, body_ja, body_en


In [4]:
# Extractor helpers for registered datasets

def parse_parallel_fiction(item, idx):
    """
    Extracts, cleans, and packages fields for ParallelFiction dataset.
    """
    meta = item.get('meta', {})
    series_title_eng = meta.get('general', {}).get('series_title_eng', 'Unknown')
    writer = meta.get('syosetu', {}).get('writer', 'Unknown')
    source = "syosetu"
    
    chapter_number, title_ja, title_en, body_ja, body_en = extract_chapter_info(item['src'], item['trg'], idx)
    cleaned_ja = clean_japanese(clean_html(body_ja))
    cleaned_en = clean_english(clean_html(body_en))
    
    return {
        "series_title": series_title_eng,
        "author": writer,
        "source": source,
        "chapter_number": chapter_number,
        "chapter_title": title_en,
        "text_ja": cleaned_ja,
        "text_en": cleaned_en,
        "text_zh": ""
    }

def parse_scribblehub(item, idx):
    """
    Extracts, cleans, and packages fields for ScribbleHub dataset (monolingual English).
    """
    meta = item.get('meta', {})
    title_str = meta.get('title', 'Unknown')
    
    if " - " in title_str:
        parts = title_str.split(" - ", 1)
        series_title = parts[0].strip()
        chapter_title = parts[1].strip()
    else:
        series_title = title_str
        chapter_title = title_str
        
    author = meta.get('author', 'Unknown')
    source = "scribblehub"
    chapter_number = parse_chapter_number(chapter_title)
    cleaned_en = clean_english(clean_html(item.get('text', '')))
    
    return {
        "series_title": series_title,
        "author": author,
        "source": source,
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
        "text_ja": "",
        "text_en": cleaned_en,
        "text_zh": ""
    }

def parse_royalroad(item, idx):
    """
    Extracts, cleans, and packages fields for RoyalRoad dataset (monolingual English).
    """
    series_title = item.get('title', 'Unknown')
    author = item.get('author', 'Unknown')
    source = "royalroad"
    chapter_title = item.get('chapter_title', 'Unknown')
    chapter_number = parse_chapter_number(chapter_title)
    cleaned_en = clean_english(clean_html(item.get('text', '')))
    
    return {
        "series_title": series_title,
        "author": author,
        "source": source,
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
        "text_ja": "",
        "text_en": cleaned_en,
        "text_zh": ""
    }

def parse_cnnovel(item, idx):
    """
    Extracts, cleans, and packages fields for RyokoAI_CNNovel125K dataset (monolingual Chinese).
    """
    meta = item.get('meta', {})
    title_str = meta.get('title', 'Unknown')
    
    if " - " in title_str:
        parts = title_str.split(" - ", 1)
        series_title = parts[0].strip()
        chapter_title = parts[1].strip()
    else:
        series_title = title_str
        chapter_title = title_str
        
    author = meta.get('author', 'Unknown')
    source = "cnnovel"
    chapter_number = parse_chapter_number(chapter_title)
    cleaned_zh = clean_html(item.get('text', ''))
    
    return {
        "series_title": series_title,
        "author": author,
        "source": source,
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
        "text_ja": "",
        "text_en": "",
        "text_zh": cleaned_zh
    }

DATASET_REGISTRY = {
    "NilanE/ParallelFiction-Ja_En-100k": {
        "extractor": parse_parallel_fiction,
    },
    "botp/RyokoAI_ScribbleHub17K": {
        "extractor": parse_scribblehub,
    },
    "OmniAICreator/RoyalRoad-1.61M": {
        "extractor": parse_royalroad,
    },
    "botp/RyokoAI_CNNovel125K": {
        "extractor": parse_cnnovel,
    }
}

# Project Gutenberg Scraper Helpers

async def download_gutenberg_async(book_id: str) -> str:
    """Downloads raw text from Project Gutenberg asynchronously using aiohttp (max 30 simultaneous connections)."""
    urls = [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt"
    ]
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    connector = aiohttp.TCPConnector(limit=30, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        for url in urls:
            try:
                async with session.get(url, headers=headers, timeout=10) as response:
                    if response.status == 200:
                        data = await response.read()
                        return data.decode('utf-8', errors='ignore')
            except Exception:
                continue
    raise ValueError(f"Could not download Gutenberg book with ID: {book_id}")

def download_gutenberg(book_id: str) -> str:
    """Wrapper to run the async Gutenberg downloader synchronously."""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        
    if loop.is_running():
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future = pool.submit(asyncio.run, download_gutenberg_async(book_id))
            return future.result()
    else:
        return loop.run_until_complete(download_gutenberg_async(book_id))

def parse_gutenberg(text: str):
    """Extracts title, author, language and splits the body into chapters."""
    title = "Unknown Gutenberg Book"
    author = "Unknown Author"
    lang = "en"

    title_match = re.search(r"Title:\s*(.*)", text)
    if title_match:
        title = title_match.group(1).strip()

    author_match = re.search(r"Author:\s*(.*)", text)
    if author_match:
        author = author_match.group(1).strip()

    lang_match = re.search(r"Language:\s*(.*)", text)
    if lang_match:
        lang_str = lang_match.group(1).strip().lower()
        if "chinese" in lang_str or "zh" in lang_str:
            lang = "zh"
        elif "japanese" in lang_str or "ja" in lang_str:
            lang = "ja"

    start_match = re.search(r"\*\*\*\s*START OF TH[IS|E] PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)
    end_match = re.search(r"\*\*\*\s*END OF TH[IS|E] PROJECT GUTENBERG EBOOK.*?\*\*\*", text, re.IGNORECASE)

    start_idx = start_match.end() if start_match else 0
    end_idx = end_match.start() if end_match else len(text)

    body = text[start_idx:end_idx].strip()

    # Split by chapter headers
    chapter_pattern = r"(?mi)^(?:\s*(?:CHAPTER|Chapter|Ch\.)\s+(?:[0-9]+|[IVXLCDM]+)\.?\s*|(?:\s*[IVXLCDM]+\.?\s*\n))"
    matches = list(re.finditer(chapter_pattern, body))

    chapters = []
    if not matches:
        chapters.append({
            "chapter_number": 1,
            "chapter_title": "Full Book",
            "text": body,
            "lang": lang
        })
    else:
        for idx, match in enumerate(matches):
            start = match.start()
            end = matches[idx + 1].start() if idx + 1 < len(matches) else len(body)

            ch_header = match.group(0).strip()
            ch_body = body[start + len(match.group(0)):end].strip()

            if not ch_body:
                continue

            ch_num = idx + 1
            num_match = re.search(r"(?:CHAPTER|Chapter|Ch\.)\s+([0-9]+|[IVXLCDM]+)", ch_header, re.IGNORECASE)
            if num_match:
                val = num_match.group(1)
                if val.isdigit():
                    ch_num = int(val)

            ch_lines = ch_body.split('\n')
            first_line = ch_lines[0].strip()
            if len(first_line) > 0 and len(first_line) < 100 and not re.search(r'[.!?]', first_line):
                ch_title = f"{ch_header}: {first_line}"
                ch_text = "\n".join(ch_lines[1:]).strip()
            else:
                ch_title = ch_header
                ch_text = ch_body

            chapters.append({
                "chapter_number": ch_num,
                "chapter_title": ch_title,
                "text": ch_text,
                "lang": lang
            })
    return title, author, chapters

In [5]:
# Baseline NLP Feature Extractor

def extract_features(text: str, lang: str = "en"):
    """
    Computes baseline features: token counts, sentence counts,
    punctuation density, and dialogue ratios.
    """
    if not text:
        return {"token_count": 0, "sentence_count": 0, "punctuation_density": 0.0, "dialogue_ratio": 0.0}
        
    if lang == "en":
        tokens = re.findall(r'\b\w+\b', text)
        token_count = len(tokens)
        sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
        sentence_count = len(sentences)
        punctuations = re.findall(r'[.,\/#!$%\^&\*;:{}=\-_`~()?"\']', text)
        punc_count = len(punctuations)
        char_count = len(text)
        punc_density = punc_count / char_count if char_count > 0 else 0.0
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        dialogue_lines = [line for line in lines if line.startswith('"') or line.startswith("'") or line.startswith('“') or line.startswith('”')]
        dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
    else: # ja or zh
        tokens = [c for c in text if not c.isspace()]
        token_count = len(tokens)
        sentences = [s for s in re.split(r'[。！？\n]+', text) if s.strip()]
        sentence_count = len(sentences)
        if lang == "zh":
            punctuations = re.findall(r'[，、。！？；：""‘’（）《》【】『』「」——……]', text)
        else: # ja
            punctuations = re.findall(r'[、。！？「」『』（）―…ー・]', text)
        punc_count = len(punctuations)
        char_count = len(text)
        punc_density = punc_count / char_count if char_count > 0 else 0.0
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        if lang == "zh":
            dialogue_lines = [line for line in lines if line.startswith('“') or line.startswith('「') or line.startswith('安全') or line.startswith('『')]
        else: # ja
            dialogue_lines = [line for line in lines if line.startswith('「') or line.startswith('『')]
        dialogue_ratio = len(dialogue_lines) / len(lines) if lines else 0.0
        
    return {
        "token_count": token_count,
        "sentence_count": sentence_count,
        "punctuation_density": punc_density,
        "dialogue_ratio": dialogue_ratio
    }

In [6]:
# ETL Orchestration

def run_etl(dataset_name: str = "NilanE/ParallelFiction-Ja_En-100k", num_records: int = 20):
    if os.path.exists("data"):
        db_path = "data/kisholens.db"
    elif os.path.exists("../data"):
        db_path = "../data/kisholens.db"
    else:
        os.makedirs("../data", exist_ok=True)
        db_path = "../data/kisholens.db"
    
    engine = create_engine(f"sqlite:///{db_path}")
    try:
        SQLModel.metadata.create_all(engine)
        print(f"Database path: {db_path}")
        novels_cache = {}
        
        # Handle Project Gutenberg path
        if dataset_name.startswith("gutenberg/"):
            book_id = dataset_name.split("/")[1]
            print(f"Downloading Gutenberg book ID: {book_id}...")
            raw_text = download_gutenberg(book_id)
            series_title, author, chapters = parse_gutenberg(raw_text)
            source = "gutenberg"
            chapters_to_ingest = chapters[:num_records]
            
            with Session(engine) as session:
                novel_key = (series_title, author)
                if novel_key not in novels_cache:
                    statement = select(Novel).where(Novel.title == series_title, Novel.author == author)
                    existing_novel = session.exec(statement).first()
                    if existing_novel:
                        novels_cache[novel_key] = existing_novel.id
                    else:
                        novel = Novel(title=series_title, author=author, source=source)
                        session.add(novel)
                        session.commit()
                        session.refresh(novel)
                        novels_cache[novel_key] = novel.id
                        print(f"Added Novel: '{series_title}' by {author} (ID: {novel.id})")
                
                novel_id = novels_cache[novel_key]
                for ch_item in chapters_to_ingest:
                    chapter_number = ch_item["chapter_number"]
                    chapter_title = ch_item["chapter_title"]
                    ch_lang = ch_item.get("lang", "en")
                    
                    raw_body = clean_html(ch_item["text"])
                    if ch_lang == "ja":
                        cleaned_text = clean_japanese(raw_body)
                    elif ch_lang == "en":
                        cleaned_text = clean_english(raw_body)
                    else:
                        cleaned_text = raw_body.strip()
                    
                    statement_ch = select(Chapter).where(Chapter.novel_id == novel_id, Chapter.chapter_number == chapter_number)
                    existing_chapter = session.exec(statement_ch).first()
                    if not existing_chapter:
                        chapter = Chapter(
                            novel_id=novel_id,
                            chapter_number=chapter_number,
                            title=chapter_title,
                            text_ja=cleaned_text if ch_lang == "ja" else "",
                            text_en=cleaned_text if ch_lang == "en" else "",
                            text_zh=cleaned_text if ch_lang == "zh" else ""
                        )
                        session.add(chapter)
                        session.commit()
                        print(f"  Ingested Chapter {chapter_number}: {chapter_title}")
                    else:
                        print(f"  Chapter {chapter_number} already ingested. Skipping DB insertion.")
                        
                    if cleaned_text:
                        feat = extract_features(cleaned_text, lang=ch_lang)
                        print(f"    Features ({ch_lang.upper()}): Tokens={feat['token_count']}, Sentences={feat['sentence_count']}, PuncDensity={feat['punctuation_density']:.3f}, DialogueRatio={feat['dialogue_ratio']:.3f}")
        
        else:
            print(f"Loading {dataset_name} in streaming mode...")
            dataset = load_dataset(dataset_name, split="train", streaming=True)
            iterator = iter(dataset)
            extractor = DATASET_REGISTRY[dataset_name]["extractor"]
            
            with Session(engine) as session:
                for idx in range(num_records):
                    try:
                        item = next(iterator)
                    except StopIteration:
                        print("Stream ran dry early.")
                        break
                    
                    parsed = extractor(item, idx)
                    series_title = parsed["series_title"]
                    author = parsed["author"]
                    source = parsed["source"]
                    chapter_number = parsed["chapter_number"]
                    chapter_title = parsed["chapter_title"]
                    cleaned_ja = parsed.get("text_ja", "")
                    cleaned_en = parsed.get("text_en", "")
                    cleaned_zh = parsed.get("text_zh", "")
                    
                    novel_key = (series_title, author)
                    if novel_key not in novels_cache:
                        statement = select(Novel).where(Novel.title == series_title, Novel.author == author)
                        existing_novel = session.exec(statement).first()
                        if existing_novel:
                            novels_cache[novel_key] = existing_novel.id
                        else:
                            novel = Novel(title=series_title, author=author, source=source)
                            session.add(novel)
                            session.commit()
                            session.refresh(novel)
                            novels_cache[novel_key] = novel.id
                            print(f"Added Novel: '{series_title}' by {author} (ID: {novel.id})")
                        
                    novel_id = novels_cache[novel_key]
                    
                    if chapter_number is None:
                        max_ch = session.exec(select(func.max(Chapter.chapter_number)).where(Chapter.novel_id == novel_id)).one()
                        chapter_number = (max_ch or 0) + 1
                    
                    if chapter_title == "Chapter " or chapter_title == "Chapter":
                        chapter_title = f"Chapter {chapter_number}"
                    
                    statement_ch = select(Chapter).where(Chapter.novel_id == novel_id, Chapter.chapter_number == chapter_number)
                    existing_chapter = session.exec(statement_ch).first()
                    if not existing_chapter:
                        chapter = Chapter(
                            novel_id=novel_id,
                            chapter_number=chapter_number,
                            title=chapter_title,
                            text_ja=cleaned_ja,
                            text_en=cleaned_en,
                            text_zh=cleaned_zh
                        )
                        session.add(chapter)
                        session.commit()
                        print(f"  Ingested Chapter {chapter_number}: {chapter_title}")
                    else:
                        print(f"  Chapter {chapter_number} already ingested. Skipping DB insertion.")
                    
                    if cleaned_ja:
                        feat_ja = extract_features(cleaned_ja, lang="ja")
                        print(f"    Features (JA): Tokens={feat_ja['token_count']}, Sentences={feat_ja['sentence_count']}, PuncDensity={feat_ja['punctuation_density']:.3f}, DialogueRatio={feat_ja['dialogue_ratio']:.3f}")
                    if cleaned_en:
                        feat_en = extract_features(cleaned_en, lang="en")
                        print(f"    Features (EN): Tokens={feat_en['token_count']}, Sentences={feat_en['sentence_count']}, PuncDensity={feat_en['punctuation_density']:.3f}, DialogueRatio={feat_en['dialogue_ratio']:.3f}")
                    if cleaned_zh:
                        feat_zh = extract_features(cleaned_zh, lang="zh")
                        print(f"    Features (ZH): Tokens={feat_zh['token_count']}, Sentences={feat_zh['sentence_count']}, PuncDensity={feat_zh['punctuation_density']:.3f}, DialogueRatio={feat_zh['dialogue_ratio']:.3f}")
    finally:
        engine.dispose()
        print(f"ETL run for {dataset_name} completed.")


In [7]:
run_etl("NilanE/ParallelFiction-Ja_En-100k", 2)

Database path: ../data/kisholens.db
Loading NilanE/ParallelFiction-Ja_En-100k in streaming mode...


  Chapter 77 already ingested. Skipping DB insertion.
    Features (JA): Tokens=3102, Sentences=137, PuncDensity=0.103, DialogueRatio=0.385
    Features (EN): Tokens=1454, Sentences=142, PuncDensity=0.057, DialogueRatio=0.385
  Chapter 36 already ingested. Skipping DB insertion.
    Features (JA): Tokens=3517, Sentences=164, PuncDensity=0.104, DialogueRatio=0.420
    Features (EN): Tokens=1710, Sentences=182, PuncDensity=0.068, DialogueRatio=0.427
ETL run for NilanE/ParallelFiction-Ja_En-100k completed.


In [8]:
run_etl("botp/RyokoAI_ScribbleHub17K", 2)

Database path: ../data/kisholens.db
Loading botp/RyokoAI_ScribbleHub17K in streaming mode...


  Ingested Chapter 2: Sir Andy
    Features (EN): Tokens=2768, Sentences=219, PuncDensity=0.034, DialogueRatio=0.109
  Ingested Chapter 2: One Night as the Queen
    Features (EN): Tokens=4115, Sentences=324, PuncDensity=0.033, DialogueRatio=0.022
ETL run for botp/RyokoAI_ScribbleHub17K completed.


In [9]:
run_etl("OmniAICreator/RoyalRoad-1.61M", 2)

Database path: ../data/kisholens.db
Loading OmniAICreator/RoyalRoad-1.61M in streaming mode...


Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

  Ingested Chapter 2: Smoke and Silence
    Features (EN): Tokens=1364, Sentences=169, PuncDensity=0.041, DialogueRatio=0.197
  Ingested Chapter 2: One
    Features (EN): Tokens=621, Sentences=57, PuncDensity=0.033, DialogueRatio=0.381
ETL run for OmniAICreator/RoyalRoad-1.61M completed.


In [10]:
run_etl("gutenberg/1342", 1)

Database path: ../data/kisholens.db


Added Novel: 'Pride and Prejudice' by Jane Austen (ID: 6)
  Ingested Chapter 1: Chapter I.: ]
    Features (EN): Tokens=893, Sentences=78, PuncDensity=0.038, DialogueRatio=0.349
ETL run for gutenberg/1342 completed.


In [11]:
run_etl("gutenberg/23950", 1)

Database path: ../data/kisholens.db


Added Novel: '三國志演義' by Guanzhong Luo (ID: 7)
  Ingested Chapter 1: Full Book
    Features (ZH): Tokens=593597, Sentences=44038, PuncDensity=0.173, DialogueRatio=0.015
ETL run for gutenberg/23950 completed.


In [12]:
run_etl("botp/RyokoAI_CNNovel125K", 2)

Database path: ../data/kisholens.db
Loading botp/RyokoAI_CNNovel125K in streaming mode...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Added Novel: '为爱入局：嫁给秦先生' by 奥德萨 (ID: 8)
  Ingested Chapter 1: 为爱入局：嫁给秦先生
    Features (ZH): Tokens=57429, Sentences=2624, PuncDensity=0.092, DialogueRatio=0.308
Added Novel: '绝世弃婿' by 绷带怪 (ID: 9)
  Ingested Chapter 1: 绝世弃婿
    Features (ZH): Tokens=75288, Sentences=3383, PuncDensity=0.093, DialogueRatio=0.281
ETL run for botp/RyokoAI_CNNovel125K completed.


In [13]:
# Verification Query

if os.path.exists("data/kisholens.db"):
    db_path = "data/kisholens.db"
else:
    db_path = "../data/kisholens.db"

engine = create_engine(f"sqlite:///{db_path}")
with Session(engine) as session:
    novels = session.exec(select(Novel)).all()
    print(f"Total novels in database: {len(novels)}")
    for n in novels:
        print(f"  - [{n.id}] {n.title} (by {n.author}) [Source: {n.source}]")
        
    expected_sources = ["syosetu", "scribblehub", "royalroad", "gutenberg", "cnnovel"]
    actual_sources = set(n.source for n in novels)
    print("\nSource Verification Check:")
    for source in expected_sources:
        if source in actual_sources:
            print(f"  [OK] Successfully retrieved novels and chapters from source: '{source}'")
        else:
            print(f"  [FAIL] No data found in database for source: '{source}'")
            
    print("\nVerifying chapter content samples and metrics from different sources:")
    for source in expected_sources:
        print(f"\nChapters from source: {source}")
        novel_ids = [n.id for n in novels if n.source == source]
        if novel_ids:
            chapters = session.exec(select(Chapter).where(Chapter.novel_id.in_(novel_ids)).limit(2)).all()
            for c in chapters:
                print(f"  - [{c.id}] Chapter {c.chapter_number}: {c.title} (Novel ID: {c.novel_id})")
                if c.text_ja:
                    print(f"    JA text (first 100 chars): {c.text_ja[:100]}...")
                    feat_ja = extract_features(c.text_ja, lang="ja")
                    print(f"    JA Metrics: Tokens={feat_ja['token_count']}, Sentences={feat_ja['sentence_count']}, PuncDensity={feat_ja['punctuation_density']:.3f}, DialogueRatio={feat_ja['dialogue_ratio']:.3f}")
                if c.text_en:
                    print(f"    EN text (first 100 chars): {c.text_en[:100]}...")
                    feat_en = extract_features(c.text_en, lang="en")
                    print(f"    EN Metrics: Tokens={feat_en['token_count']}, Sentences={feat_en['sentence_count']}, PuncDensity={feat_en['punctuation_density']:.3f}, DialogueRatio={feat_en['dialogue_ratio']:.3f}")
                if hasattr(c, 'text_zh') and c.text_zh:
                    print(f"    ZH text (first 100 chars): {c.text_zh[:100]}...")
                    feat_zh = extract_features(c.text_zh, lang="zh")
                    print(f"    ZH Metrics: Tokens={feat_zh['token_count']}, Sentences={feat_zh['sentence_count']}, PuncDensity={feat_zh['punctuation_density']:.3f}, DialogueRatio={feat_zh['dialogue_ratio']:.3f}")
        else:
            print("  No novels found for this source.")
engine.dispose()

Total novels in database: 9
  - [1] Noble Reincarnation~Blessed With the Strongest Power From Birth (by 三木なずな) [Source: syosetu]
  - [2] Sir Andy (by mrsimple) [Source: scribblehub]
  - [3] One Night as the Queen (by mrsimple) [Source: scribblehub]
  - [4] Threadbare Titans (by Pythonogram#) [Source: royalroad]
  - [5] Skysea (by MistOverSnow) [Source: royalroad]
  - [6] Pride and Prejudice (by Jane Austen) [Source: gutenberg]
  - [7] 三國志演義 (by Guanzhong Luo) [Source: gutenberg]
  - [8] 为爱入局：嫁给秦先生 (by 奥德萨) [Source: cnnovel]
  - [9] 绝世弃婿 (by 绷带怪) [Source: cnnovel]

Source Verification Check:
  [OK] Successfully retrieved novels and chapters from source: 'syosetu'
  [OK] Successfully retrieved novels and chapters from source: 'scribblehub'
  [OK] Successfully retrieved novels and chapters from source: 'royalroad'
  [OK] Successfully retrieved novels and chapters from source: 'gutenberg'
  [OK] Successfully retrieved novels and chapters from source: 'cnnovel'

Verifying chapter content sa